## 01 — Data Ingestion

**Project:** EV Battery Capacity Prediction  
**Notebook purpose:** verify the raw dataset's structure, identify the sensor
channels, and build a reusable car-to-files index.

---

## Dataset

**EVBattery** — He, Wang, Sun et al. (2023), *Real-World Electric Vehicle Battery
Dataset*, NeurIPS 2023 Datasets & Benchmarks Track. Released under CC-BY-NC-SA.

Real-world charging telemetry collected from public charging stations, covering a
fleet of production electric vehicles. This project uses the brand 1 release.

## Reference paper

van den Hoven & Ranković (2026), *Data-Driven Battery Capacity Estimation in
Electric Vehicles: Insights from Large-Scale Real-World Data*, **Energy Systems**.  
https://link.springer.com/article/10.1007/s12667-025-00775-y

Their study benchmarks ARIMA(X), XGBoost, LSTM and TCN on this data. Best result:
**LSTM, RMSE 1.42 / MAE 1.09 / MAPE 2.70%**. Those figures are the benchmark this
project measures against.

---

## What this notebook produces

`data/processed/dataset_index.json` — a mapping from car number to that car's
snippet file paths, plus per-car metadata. Every downstream notebook loads this
instead of rescanning the raw files.

## 1. Setup

Expected directory layout:

```
ev-battery-capacity-prediction/
├── data/
│   ├── raw/battery_dataset1/
│   │   ├── data/     # one .pkl per charging snippet
│   │   └── label/    # label.csv: car number -> fault flag
│   └── processed/    # index written here
└── notebooks/        # this file
```

In [1]:
import json
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

np.set_printoptions(precision=3, suppress=True)

RAW_DIR = Path("../data/raw/battery_dataset1")
DATA_DIR = RAW_DIR / "data"
INDEX_DIR = Path("../data/processed")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("data dir exists :", DATA_DIR.exists())
print("label dir exists:", (RAW_DIR / "label").exists())

data dir exists : True
label dir exists: True


## 2. What is in a single file

Filenames are arbitrary integers (`178799.pkl`). They encode **nothing** — not the
car, not the date, not the session. All identifying information lives inside the
file. This fact drives the rest of the notebook.

Per the dataset's own documentation, each `.pkl` is a **2-element tuple**:

| Element | Contents |
|---|---|
| `record[0]` | numpy array of charging sensor readings |
| `record[1]` | `OrderedDict` of metadata |

In [2]:
# Pick any file to inspect the structure.
# weights_only=False is required: these pickles hold arbitrary Python objects
# (a tuple containing a dict), not model weights.

sample_file = sorted(DATA_DIR.glob("*.pkl"))[0]
record = torch.load(sample_file, weights_only=False)

print("file :", sample_file.name)
print("type :", type(record))
print("parts:", len(record))

file : 0.pkl
type : <class 'tuple'>
parts: 2


In [3]:
data_part, meta_part = record

print("=== PART 1 — time series ===")
print("type :", type(data_part))
print("shape:", data_part.shape)
print()
print("=== PART 2 — metadata ===")
for k, v in meta_part.items():
    print(f"  {k:<16} {v}")

=== PART 1 — time series ===
type : <class 'numpy.ndarray'>
shape: (128, 8)

=== PART 2 — metadata ===
  label            00
  car              0
  charge_segment   1
  mileage          87968.496
  capacity         0


### Structure confirmed

**Part 1 — `(128, 8)` numpy array.** 128 timesteps × 8 sensor channels. One row per
reading, 10 seconds apart, so a snippet covers roughly **21 minutes** of charging.

**Part 2 — metadata:**

| Key | Role in this project |
|---|---|
| `capacity` | **The prediction target.** Battery capacity in Ah. |
| `car` | Vehicle ID. **The grouping key for the train/test split.** |
| `mileage` | Odometer reading (km) at the time of this snippet. |
| `charge_segment` | Charging-session ID. Ties sibling snippets together. |
| `label` | Fault flag from the dataset's anomaly-detection task. Not used here. |

> **Two different "labels".** The `label` field is a binary healthy/faulty flag
> belonging to the dataset's *other* task (anomaly detection). It answers "is
> something wrong?". This project answers "how degraded is it?" — a different
> question, whose target is `capacity`. A battery at 80% capacity is worn but not
> faulty.

## 3. Session vs. snippet

One **charging session** is one plug-in. During it, sensors log a reading every 10
seconds, producing a table of arbitrary length.

Models need fixed-size input, so each session is cut into overlapping **128-step
windows** by a sliding window. Each window is saved as its own `.pkl`.

```
session rows:  0 ────────────────────────────── N

window 1:      [0 ....... 127]
window 2:            [50 ....... 177]
window 3:                  [100 ....... 227]
```

So **one session → several snippets**, and `charge_segment` is what marks them as
siblings.

**Why this matters:** overlapping windows are near-identical. If one lands in the
training set and its neighbour in the test set, the model has effectively seen the
answer. This is the leakage risk that dictates the splitting strategy in notebook 03.

In [4]:
# How many real charging sessions underlie a given number of snippets?

probe = sorted(DATA_DIR.glob("*.pkl"))[:500]
segments, cars = set(), set()

for p in tqdm(probe, desc="probing"):
    _, meta = torch.load(p, weights_only=False)
    cars.add(meta["car"])
    segments.add((meta["car"], meta["charge_segment"]))

print(f"\n500 snippets → {len(segments)} distinct charging sessions "
      f"across {len(cars)} cars")
print(f"≈ {500 / len(segments):.1f} snippets per session")

probing: 100%|██████████| 500/500 [00:00<00:00, 2628.76it/s]


500 snippets → 105 distinct charging sessions across 3 cars
≈ 4.8 snippets per session


### Finding — roughly 10 snippets per charging session

A car's several-thousand snippets therefore come from only a few hundred genuinely
independent charging events. The effective sample size is smaller than the raw
snippet count suggests — another reason the split must be made at the vehicle level.

## 4. Identifying the 8 sensor channels

The columns arrive unnamed. Their identity can be recovered from their value ranges,
cross-referenced against the feature list given in the reference paper: average cell
voltage, charging current, max/min cell voltage, max/min cell temperature, state of
charge, and timestamp.

In [6]:
print("Per-channel min / max for one snippet:\n")
for i in range(8):
    lo, hi = data_part[:, i].min(), data_part[:, i].max()
    print(f"  col {i}:  min {lo:>10.3f}   max {hi:>10.3f}")

Per-channel min / max for one snippet:

  col 0:  min      3.761   max      3.957
  col 1:  min    -64.300   max    -42.700
  col 2:  min     22.800   max     50.233
  col 3:  min      3.809   max      3.998
  col 4:  min      3.625   max      3.720
  col 5:  min     11.000   max     17.000
  col 6:  min      4.000   max     12.000
  col 7:  min      0.000   max   1270.000


### Channel map

| Col | Feature | Typical range | How it was identified |
|---|---|---|---|
| 0 | Average cell voltage | ~3.7 – 4.2 V | Li-ion cell voltage band |
| 1 | Charging current | negative, ~−14 A | Negative sign = charging convention |
| 2 | State of charge (SOC) | 0 – 100 % | Only 0–100 bounded channel |
| 3 | Max cell voltage | slightly above col 0 | Sits just above the average |
| 4 | Min cell voltage | slightly below col 0 | Sits just below the average |
| 5 | Max cell temperature | ambient °C | Paired with col 6, integer-valued |
| 6 | Min cell temperature | ambient °C | Always ≤ col 5 |
| 7 | Timestamp | 0 → 1270 s | Monotonic, 10 s steps, 128 steps |

The giveaway for cols 0/3/4 is that they sit almost on top of each other with
`min < avg < max` — three statistics of the same quantity.

### Why max and min are separate channels

An EV pack is hundreds of individual cells wired together. Ideally they sit at the
same voltage. As the pack ages they drift apart, and that spread — **cell
imbalance** — is a recognised degradation indicator.

So `max_V − min_V` and `max_T − min_T` are physically meaningful derived features,
potentially carrying more signal than the three highly-correlated raw voltages.
Noted as a feature-engineering option for the modelling stage.

In [7]:
# Cell imbalance for this snippet
v_spread = (data_part[:, 3] - data_part[:, 4]).mean()
t_spread = (data_part[:, 5] - data_part[:, 6]).mean()

print(f"Mean cell voltage spread     : {v_spread:.4f} V")
print(f"Mean cell temperature spread : {t_spread:.2f} °C")

Mean cell voltage spread     : 0.2208 V
Mean cell temperature spread : 4.80 °C


## 5. Why an index is needed

To train on a chosen set of vehicles, the files belonging to each vehicle must be
identifiable. Since filenames carry no car information, the **only** way to group
files by car is to open each one and read its metadata.

Doing that on every run would be prohibitively slow. Instead it is done **once**, and
the result is saved:

```
car 2   → [12045.pkl, 88213.pkl, 40021.pkl, ...]
car 43  → [178799.pkl, 220514.pkl, ...]
```

### The `capacity == 0` filter

Capacity is not measurable on every charging session. The measurement requires the
session to pass through a specific voltage window at a steady current. Sessions that
never satisfy those conditions are stored with `capacity = 0` as a placeholder — not
a real reading of zero.

Those snippets cannot be used for supervised training and are excluded from the index.

In [8]:
# ---------------------------------------------------------------------------
# Build the car → files index.
#
# Opens every .pkl once and reads metadata only (the 128x8 array is discarded,
# which keeps the scan as fast as possible).
#
# Produces:
#   car_to_files : car number -> list of file paths (capacity-labelled only)
#   car_info     : car number -> fault label + mileage range
#
# Runtime ~2-3 min. One-time cost; saved to JSON in the next cell.
# ---------------------------------------------------------------------------

all_files = sorted(DATA_DIR.glob("*.pkl"))
print(f"Scanning {len(all_files):,} files...")

car_to_files = defaultdict(list)
car_info = {}
skipped = 0

for f in tqdm(all_files):
    _, meta = torch.load(f, weights_only=False)

    if meta["capacity"] == 0:          # no valid capacity measurement
        skipped += 1
        continue

    car = meta["car"]
    car_to_files[car].append(str(f))

    # running min/max of mileage, which grows over each car's life
    if car not in car_info:
        car_info[car] = {
            "label": meta["label"],
            "min_mileage": meta["mileage"],
            "max_mileage": meta["mileage"],
        }
    else:
        info = car_info[car]
        info["min_mileage"] = min(info["min_mileage"], meta["mileage"])
        info["max_mileage"] = max(info["max_mileage"], meta["mileage"])

n_labelled = sum(len(v) for v in car_to_files.values())

print(f"\nTotal files            : {len(all_files):,}")
print(f"Cars with labelled data: {len(car_to_files)}")
print(f"Labelled snippets      : {n_labelled:,}")
print(f"Skipped (capacity == 0): {skipped:,}  ({skipped / len(all_files):.1%})")

Scanning 629,121 files...


100%|██████████| 629121/629121 [02:27<00:00, 4259.63it/s]


Total files            : 629,121
Cars with labelled data: 100
Labelled snippets      : 349,741
Skipped (capacity == 0): 279,380  (44.4%)


### Scan results

| | Count |
|---|---|
| Total `.pkl` files | 629,121 |
| Usable (capacity labelled) | **349,741** |
| Unusable (`capacity == 0`) | 279,380 — **44%** |
| Vehicles with usable data | **100** |

### Finding — 44% of real-world charging data carries no label

This is the single most important fact about the dataset, and it is the project's
motivation rather than a data-quality problem.

Capacity is measured by charging through a fixed voltage window (3.77 V → 4.05 V) at
a steady current and integrating the charge that flows in. Three conditions must all
hold:

1. Charging starts below 3.77 V
2. Charging continues past 4.05 V
3. Current holds steady throughout

Real drivers break all three routinely — plugging in at 60% charge, unplugging early,
using whatever current the station provides, or the vehicle throttling current as the
pack warms. **None of these are errors. They are normal usage.**

But the charging *behaviour* is recorded either way. If a model can read battery
health from that behaviour, health becomes observable on **every** charge rather than
the 56% that happen to satisfy the measurement protocol. That is the deployable
capability this project targets.

## 6. Persist the index

Saved as JSON so every later notebook loads in milliseconds instead of repeating the
scan. JSON keys must be strings, so car numbers are converted.

In [9]:
index = {
    "car_to_files": {str(k): v for k, v in car_to_files.items()},
    "car_info": {str(k): v for k, v in car_info.items()},
    "meta": {
        "total_files": len(all_files),
        "labelled_snippets": n_labelled,
        "skipped_unlabelled": skipped,
        "n_cars": len(car_to_files),
    },
}

index_path = INDEX_DIR / "dataset_index.json"
with open(index_path, "w") as fp:
    json.dump(index, fp)

print(f"Saved → {index_path}")
print(f"Size  : {index_path.stat().st_size / 1024**2:.1f} MB")

Saved → ../data/processed/dataset_index.json
Size  : 16.0 MB


## 7. Per-car summary table

Merges the two index dictionaries into one row per vehicle. This table is the input
to the subset and split decisions.

In [10]:
cars_df = pd.DataFrame([
    {
        "car": int(car),
        "n_snippets": len(files),
        "label": car_info[car]["label"],
        "min_mileage": car_info[car]["min_mileage"],
        "max_mileage": car_info[car]["max_mileage"],
        "mileage_span": car_info[car]["max_mileage"] - car_info[car]["min_mileage"],
    }
    for car, files in car_to_files.items()
]).sort_values("car").reset_index(drop=True)

cars_df.to_csv(INDEX_DIR / "car_summary.csv", index=False)

print(cars_df.describe())
cars_df.head(15)

              car   n_snippets    min_mileage    max_mileage   mileage_span
count  100.000000   100.000000     100.000000     100.000000     100.000000
mean    87.450000  3497.410000   66399.777312  173522.167104  107122.389792
std     55.341981  1978.646441   36760.196080   33090.636020   41165.995966
min      2.000000   390.000000       0.000000   41923.200000   14023.680000
25%     37.500000  1807.500000   53441.520000  159659.042400   85434.069600
50%     80.500000  3650.000000   77559.028800  171544.560000   99236.544000
75%    136.750000  4755.500000   88924.149600  191137.821600  121323.813600
max    187.000000  9101.000000  180763.862400  267515.529600  256047.369600


,car,n_snippets,label,min_mileage,max_mileage,mileage_span
0,2,5556,00,1662.1440,187433.4528,185771.3088
1,3,3729,00,100294.7616,213731.6544,113436.8928
2,4,5222,00,214.7904,193132.1568,192917.3664
3,5,2784,00,58428.4800,158983.1232,100554.6432
4,6,3366,00,73296.7488,159770.4768,86473.7280
5,7,1782,00,88630.5024,192568.5696,103938.0672
6,11,4453,10,104135.4336,233019.4944,128884.0608
7,14,6290,00,6969.6000,159933.8400,152964.2400
8,16,7086,00,3919.8720,178666.3296,174746.4576
9,17,1693,00,0.0000,178304.4384,178304.4384


In [11]:
print(cars_df["label"].value_counts())
print()
print(f"Total snippets across all {len(cars_df)} cars: "
      f"{cars_df['n_snippets'].sum():,}")

label
00    96
10     4
Name: count, dtype: int64

Total snippets across all 100 cars: 349,741


### Findings from the per-car table

**A — Snippet counts are severely imbalanced.**  
Range 390 to 9,101 per car (mean 3,497, std 1,979) — a **23× spread**.

Consequences: high-volume cars contribute disproportionately more gradient updates,
so the model will implicitly fit them better; and a low-volume car in the test set may
score poorly for reasons unrelated to model quality. **Metrics must be reported
per-car, not only in aggregate.**

The reference paper hit exactly this — a single vehicle drove roughly 70% of their
worst-error cases, and removing it moved their headline metrics by about 5%.

**B — Each vehicle is tracked across a long lifespan.**  
Median car runs 77,559 → 171,545 km: roughly **94,000 km of tracked driving**.

This is what makes the task viable at all. Battery degradation accumulates over tens
of thousands of kilometres. A dataset of one-off snapshots would contain no fade to
learn from. This one follows vehicles longitudinally, so genuine degradation is
captured.

**C — The full wear spectrum is present.**  
`min_mileage` reaches 0 (at least one vehicle tracked from new) and `max_mileage`
reaches 267,516 km (at least one tracked to heavy wear).

**D — Mileage is a per-snippet value, not a per-car property.**  
Nearly every car spans low, medium and high mileage individually. **Therefore cars
cannot be stratified by mileage.** The reference paper's per-mileage-band analysis is
a *snippet-level* evaluation performed after prediction, not a split criterion.

**E — Fault labels are sparse and not the target.**  
96 vehicles flagged healthy, 4 flagged faulty. Not stratified on: too few positives,
and irrelevant to a capacity regression.

## Summary

| Item | Value |
|---|---|
| Total files | 629,121 |
| Usable snippets | 349,741 |
| Unusable (`capacity == 0`) | 279,380 (44%) |
| Vehicles | 100 |
| Snippet shape | (128, 8) |
| Snippets per charging session | ~10 |
| Target | `capacity` (Ah) |
| Split key | `car` |

**Artefacts written:**

- `data/processed/dataset_index.json` — car → file paths, plus per-car metadata
- `data/processed/car_summary.csv` — per-vehicle summary table

**Next:** `02_eda.ipynb` — target distribution, degradation analysis, and the
baselines the sequence models must beat.